# A Catholic Introduction to Artificial Intelligence
## Final Project — Module 5: Governance, Policy & Deployment Decisions
### Classes 9 & 10

---

> *"For AI to respect human dignity and truly serve the common good, responsibility must be clearly defined at every stage: from those who design and develop these systems to those who use them and rely on them for concrete decisions. In many cases, however, the internal processes leading to a result remain opaque, making it harder to assign responsibility and correct errors. This is where accountability becomes crucial: the possibility of identifying who must 'account' for decisions, justify them, monitor them, and, when necessary, challenge them and remedy any harm caused."*
> — Pope Leo XIV, *Magnifica Humanitas*, no. 105

---

## Module Overview

You have now built a predictive AI model, evaluated it honestly, and audited it for bias. You have found that it is 81% accurate overall, catches only about 6–43% of late submissions depending on the threshold, performs significantly worse for students with weaker academic histories, and concentrates its failures on the very students most likely to need support.

The technical work is done. Now comes the harder question: **given all of this, what should be done?**

This module shifts from building to governing. You will:
1. Analyze the stakeholder landscape — who is affected by this system and how
2. Explore different deployment scenarios and evaluate each
3. Draft a formal **Data Governance Policy** for the system
4. Apply Catholic social teaching principles to each policy decision
5. Produce the core written component of your final project

This module produces the most important document in the entire project: a written governance policy that reflects both technical understanding and moral reasoning. It is what separates responsible AI development from irresponsible AI deployment.

---

## Setup: Rebuild the Full Pipeline

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (11, 5)

# ── Load and prepare ──────────────────────────────────────────────────
df = pd.read_csv('synthetic_homework_dataset.csv',
                 parse_dates=['date_assigned', 'date_submitted'])

df['time_pressure'] = df['difficulty'] / df['days_until_due']
le = LabelEncoder()
df['assignment_type_enc'] = le.fit_transform(df['assignment_type'])

FULL_FEATURES = [
    'num_questions', 'difficulty', 'days_until_due', 'time_pressure',
    'assignment_type_enc', 'prior_completion_rate',
    'prior_avg_grade', 'prior_avg_homework_time'
]
TARGET = 'completed_on_time'

X = df[FULL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
model  = LogisticRegression(random_state=42, max_iter=1000)
model.fit(scaler.fit_transform(X_train), y_train)

y_pred = model.predict(scaler.transform(X_test))
y_prob = model.predict_proba(scaler.transform(X_test))[:, 1]

results = df.loc[X_test.index].copy()
results['predicted']    = y_pred
results['prob_on_time'] = y_prob.round(4)
results['correct']      = (results['predicted'] == results[TARGET]).astype(int)
results['prior_tier']   = pd.cut(
    results['prior_completion_rate'],
    bins=[0, 0.60, 0.75, 0.90, 1.01],
    labels=['<60%', '60–75%', '75–90%', '>90%']
)

overall_acc = accuracy_score(y_test, y_pred)
print(f'Pipeline rebuilt. Overall accuracy: {overall_acc*100:.1f}%')
print(f'Test set: {len(results)} predictions')

Pipeline rebuilt. Overall accuracy: 86.0%
Test set: 100 predictions


---

## Part 1: Stakeholder Analysis — Who Is Affected?

Before writing a governance policy, we need to identify every party who has a stake in how this system operates. In AI governance, a **stakeholder** is anyone who is affected by the system's outputs — not just the people who built it or deployed it.

Good governance requires understanding each stakeholder's interests, vulnerabilities, and the specific harms they might experience from the system operating badly.

### Stakeholder Map

| Stakeholder | Their interest | What they risk from poor AI governance |
|---|---|---|
| **Student (flagged correctly)** | Receive appropriate support | Privacy violation; stigma; being defined by past performance |
| **Student (wrongly flagged)** | Not be labeled unfairly | Unwarranted scrutiny; teacher bias; record notation |
| **Student (missed — actually late)** | Receive support they need | No early help; continues to fall behind; system fails them silently |
| **Teacher / instructor** | Make better-informed decisions | Over-reliance; deskilling; accountability displaced to algorithm |
| **School administration** | Improve outcomes; allocate resources | Legal liability; reputational damage; compliance risk |
| **Parents / guardians** | Know how their child is being assessed | Hidden algorithmic assessment without knowledge or consent |
| **Students in low-prior-rate tier** | Be treated fairly despite past history | Systematic underservice; feedback loop entrenching disadvantage |
| **Future students** | Not inherit a biased system | Training data that encodes today's patterns into future predictions |

### The Most Vulnerable Stakeholder

The most vulnerable stakeholder in this system is the student who:
- Has a lower prior completion rate (and is therefore hardest for the model to serve accurately)
- Is in a difficult personal or academic situation that the model cannot detect
- Has no voice in how the model is configured or whose predictions they can challenge

*Magnifica Humanitas* (no. 78) specifically warns that social justice must begin with the most vulnerable, and that structures which produce inequality almost automatically require both personal and social conversion.

> 🤔 **Think about it:** If a student's parents discovered that an AI system had flagged their child as "at risk" without their knowledge — and that flag had influenced how their teacher interacted with their child — what would a just response look like? What would the school owe them?

---

## Part 2: Deployment Scenarios

Not all deployments of this model are equivalent. The same AI predictions, used in different ways by different people with different access levels, can produce very different outcomes. Let's examine four concrete deployment scenarios.

In [ ]:
# Visualize what the model flags at different thresholds
thresholds = [0.50, 0.70, 0.80, 0.85, 0.90]

flagged_counts, late_caught_counts, false_flag_counts = [], [], []
total_late = (y_test == 0).sum()

for thresh in thresholds:
    flagged       = (y_prob < thresh).sum()
    late_caught   = ((y_prob < thresh) & (y_test == 0)).sum()
    false_flags   = ((y_prob < thresh) & (y_test == 1)).sum()
    flagged_counts.append(flagged)
    late_caught_counts.append(late_caught)
    false_flag_counts.append(false_flags)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: how many students flagged at each threshold
axes[0].plot(thresholds, flagged_counts, 'o-', color='#5C8BC7', linewidth=2, label='Total flagged')
axes[0].plot(thresholds, late_caught_counts, 's-', color='#4CAF50', linewidth=2, label='Actually late (caught)')
axes[0].plot(thresholds, false_flag_counts, '^-', color='#E57373', linewidth=2, label='On-time (wrongly flagged)')
axes[0].axhline(total_late, color='gray', linestyle='--', linewidth=1, label=f'Total actual late = {total_late}')
axes[0].set_xlabel('Prediction Threshold (flag if prob < threshold)')
axes[0].set_ylabel('Number of Students')
axes[0].set_title('Who Gets Flagged at Each Threshold?', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xticks(thresholds)

# Right: precision vs. recall for the late class
precisions = [lc / fc if fc > 0 else 0
              for lc, fc in zip(late_caught_counts, flagged_counts)]
recalls    = [lc / total_late for lc in late_caught_counts]

axes[1].plot(recalls, precisions, 'o-', color='#7986CB', linewidth=2)
for i, thresh in enumerate(thresholds):
    axes[1].annotate(f'  {thresh}', (recalls[i], precisions[i]), fontsize=9, color='gray')
axes[1].set_xlabel('Recall — fraction of late students caught')
axes[1].set_ylabel('Precision — fraction of flagged students truly late')
axes[1].set_title('Precision vs. Recall at Different Thresholds', fontweight='bold')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print('Threshold | Flagged | Late caught | False flags | Precision | Recall')
print('-' * 72)
for t, fc, lc, ff, p, r in zip(thresholds, flagged_counts, late_caught_counts,
                                 false_flag_counts, precisions, recalls):
    print(f'{t:<10.2f}  {fc:<9}  {lc:<12}  {ff:<13}  {p:<11.2f}  {r:.2f}')

### The Four Deployment Scenarios

Read each scenario carefully. For each one, the model produces the same predictions — but the governance arrangements around those predictions are very different.

In [ ]:
# Visualize the four scenarios
scenarios = {
    'Scenario A\n"Dashboard"': {
        'description': 'Teacher sees a dashboard with all students ranked by predicted\n'
                       'risk. No threshold — all probabilities visible. Teacher uses\n'
                       'this to decide who to check in with.',
        'access': 'Teacher only',
        'threshold': 'None — full probability shown',
        'student_notified': False,
        'appeal_mechanism': False,
        'human_override': True,
    },
    'Scenario B\n"Alert"': {
        'description': 'System flags students below 0.70 probability. Teacher receives\n'
                       'an automated alert listing flagged names. Teacher must document\n'
                       'action taken for each flag.',
        'access': 'Teacher + admin log',
        'threshold': '0.70 (flags 12% of students)',
        'student_notified': False,
        'appeal_mechanism': False,
        'human_override': True,
    },
    'Scenario C\n"Transparent"': {
        'description': 'Students and families see their own risk scores. Teacher sees all.\n'
                       'Students flagged below 0.70 receive an automated outreach message.\n'
                       'Students may request removal from flagging.',
        'access': 'Student (own score) + teacher (all)',
        'threshold': '0.70',
        'student_notified': True,
        'appeal_mechanism': True,
        'human_override': True,
    },
    'Scenario D\n"Auto-action"': {
        'description': 'System automatically enrolls students below 0.85 in a remediation\n'
                       'program, sends alerts to parents, and flags the record. No human\n'
                       'review before action is taken.',
        'access': 'Automated — teacher, admin, parent notified after',
        'threshold': '0.85 (flags 39% of students)',
        'student_notified': True,
        'appeal_mechanism': False,
        'human_override': False,
    },
}

print('=== The Four Deployment Scenarios ===\n')
for name, s in scenarios.items():
    clean = name.replace('\n', ' ')
    print(f'{clean}')
    print(f'  Description:       {s["description"].split(chr(10))[0]}')
    print(f'  Threshold:         {s["threshold"]}')
    print(f'  Access:            {s["access"]}')
    print(f'  Student notified:  {s["student_notified"]}')
    print(f'  Appeal mechanism:  {s["appeal_mechanism"]}')
    print(f'  Human override:    {s["human_override"]}')
    print()

In [ ]:
# Visualize scenario comparison on key governance dimensions
fig, ax = plt.subplots(figsize=(12, 6))

dimensions = [
    'Student\nnotified',
    'Appeal\nmechanism',
    'Human\noverride required',
    'Transparent\nthreshold',
    'Action\nautomated',
    'Record\nimpact',
]

scenario_scores = {
    'A: Dashboard':   [0, 0, 1, 0, 0, 0],
    'B: Alert':       [0, 0, 1, 1, 0, 1],
    'C: Transparent': [1, 1, 1, 1, 0, 0],
    'D: Auto-action': [1, 0, 0, 1, 1, 1],
}

colors = ['#7986CB', '#81C784', '#4CAF50', '#E57373']
x = np.arange(len(dimensions))
width = 0.18

for i, (label, scores) in enumerate(scenario_scores.items()):
    bars = ax.bar(x + i * width, scores, width, label=label,
                  color=colors[i], alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(dimensions, fontsize=10)
ax.set_yticks([0, 1])
ax.set_yticklabels(['No / False', 'Yes / True'], fontsize=9)
ax.set_title('Governance Dimensions Across Four Deployment Scenarios\n'
             '(1 = present, 0 = absent)', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, 1.4)

plt.tight_layout()
plt.show()

### Evaluating the Scenarios

Each scenario makes different trade-offs between utility and protection. Use the following questions to evaluate them before writing your policy.

**Scenario A — Dashboard:** The teacher retains full judgment. But with no threshold and no student awareness, the model invisibly shapes which students receive attention — without any accountability mechanism if those choices are wrong.

**Scenario B — Alert:** Adds documentation (admin log) and a threshold, but students still have no knowledge of or recourse against the system affecting how their teacher treats them.

**Scenario C — Transparent:** Students see their own scores and can request removal. This is more respectful of dignity and autonomy — but raises questions about whether seeing a low risk score might itself harm a student's confidence or teacher expectations.

**Scenario D — Auto-action:** The most intrusive. 39% of students are automatically enrolled in remediation and have their records flagged — without any human judgment applied before the action. The model's errors (28 false flags at the 0.85 threshold) directly become adverse actions against innocent students.

> 📝 **Reflection:** *Magnifica Humanitas* (no. 102) warns that consequential decisions about individuals risk being fully delegated to automated systems that do not know compassion, mercy, forgiveness, or the hope that people are able to change. Which of the four scenarios most violates this principle? Which is closest to satisfying it? Write your ranking and reasoning below.

### Your Scenario Ranking

Rank the four scenarios from most to least acceptable, and explain your reasoning. For each, identify which Catholic principle (human dignity, subsidiarity, solidarity, common good, etc.) most directly applies.

*[Write your ranking and reasoning here]*

---

## Part 3: The Eight Elements of a Data Governance Policy

A **data governance policy** is a formal document that specifies how an AI system will be operated, who has what access, what protections exist for affected individuals, and under what conditions the system may be modified or shut down.

Good governance policies address eight elements. We will work through each one, generating data to inform the decision before writing the policy language.

| Element | What It Covers |
|---|---|
| 1. Purpose and scope | What is the system for? What decisions may it influence? |
| 2. Access and permissions | Who sees what? Under what conditions? |
| 3. Threshold and confidence requirements | At what prediction confidence level does an action occur? |
| 4. Human oversight requirements | What human review must occur before any action? |
| 5. Disclosure and transparency | What must students, families, and staff be told? |
| 6. Appeal and redress mechanisms | How can affected individuals challenge a prediction? |
| 7. Monitoring and audit schedule | How often is the model evaluated for accuracy and bias? |
| 8. Sunset and revocation conditions | Under what circumstances must the system be suspended? |

In [ ]:
# Element 3: Threshold analysis — what does each confidence level mean?
# This data should inform the policy's threshold decision.

print('=== Threshold Decision Support ===')
print('What does each confidence level mean in practice?')
print()

bands = [
    (0.00, 0.50, 'Very high risk (prob < 0.50)'),
    (0.50, 0.70, 'High risk (0.50–0.70)'),
    (0.70, 0.85, 'Moderate risk (0.70–0.85)'),
    (0.85, 1.01, 'Low risk (prob ≥ 0.85)'),
]

print(f'{'Band':<36} {'n':>5} {'Actually late':>14} {'Precision':>12} {'Pct of all late':>16}')
print('-' * 84)
total_late = (y_test == 0).sum()

for lo, hi, label in bands:
    mask = (y_prob >= lo) & (y_prob < hi)
    n = mask.sum()
    if n == 0:
        continue
    actually_late = ((y_test == 0) & mask).sum()
    precision = actually_late / n
    pct_of_late = actually_late / total_late
    print(f'{label:<36} {n:>5} {actually_late:>8}/{n:<5} {precision:>10.1%} {pct_of_late:>14.1%}')

print()
print(f'Total actually late in test set: {total_late}')
print()
print('Policy question: At what threshold should human review be required?')
print('Policy question: At what threshold should students be notified?')
print('Policy question: Should any automated action be permitted without human review?')

In [ ]:
# Visualize the risk band analysis
band_labels = ['Very high risk\n(prob < 0.50)', 'High risk\n(0.50–0.70)',
               'Moderate risk\n(0.70–0.85)', 'Low risk\n(≥ 0.85)']
band_totals, band_late, band_prec = [], [], []

for lo, hi, _ in bands:
    mask = (y_prob >= lo) & (y_prob < hi)
    n = mask.sum()
    lat = ((y_test == 0) & mask).sum()
    band_totals.append(n)
    band_late.append(lat)
    band_prec.append(lat / n if n > 0 else 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked composition of each band
ontime_vals = [t - l for t, l in zip(band_totals, band_late)]
x = np.arange(len(band_labels))
axes[0].bar(x, band_late, color='#E57373', label='Actually late', width=0.55)
axes[0].bar(x, ontime_vals, bottom=band_late, color='#81C784', label='Actually on time', width=0.55)
axes[0].set_xticks(x)
axes[0].set_xticklabels(band_labels, fontsize=9)
axes[0].set_ylabel('Number of Students')
axes[0].set_title('Who Is in Each Risk Band?\n(Color = actual outcome)', fontweight='bold')
axes[0].legend(fontsize=9)
for xi, (t, l) in enumerate(zip(band_totals, band_late)):
    if t > 0:
        axes[0].text(xi, t + 0.2, f'n={t}', ha='center', fontsize=9)

# Right: precision in each band
colors = ['#E57373', '#FF8A65', '#FFB74D', '#81C784']
axes[1].bar(x, [p * 100 for p in band_prec], color=colors, width=0.55)
axes[1].set_xticks(x)
axes[1].set_xticklabels(band_labels, fontsize=9)
axes[1].set_ylabel('Precision (%)')
axes[1].set_title('What Fraction of Each Band Is Truly Late?\n(Precision by risk band)', fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].axhline(50, color='black', linestyle='--', linewidth=1, label='50% line')
axes[1].legend(fontsize=9)
for xi, p in enumerate(band_prec):
    axes[1].text(xi, p * 100 + 1.5, f'{p*100:.0f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### Reading the Risk Band Analysis

The right chart reveals the model's actual reliability in each band:

- **Very high risk** (prob < 0.50): 50% precision — even in the highest-alert band, half the flagged students are on time
- **High risk** (0.50–0.70): 50% precision — still a coin flip
- **Moderate risk** (0.70–0.85): 19% precision — 4 in 5 flagged students are on time
- **Low risk** (≥ 0.85): 5% precision — 19 in 20 are on time (these are correctly not flagged)

This means that even in the two highest-risk bands, the model is wrong about at least half the students it flags. Any automated action based on these predictions would cause harm to a large majority of the students affected.

This is the empirical foundation for a key policy principle: **no automated adverse action should be triggered by model predictions alone.**

In [ ]:
# Element 7: What should ongoing monitoring look like?
# This code simulates what a monitoring report might track over time.

print('=== Monitoring Dashboard Prototype ===')
print('These are the metrics that should be tracked each term:')
print()

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
total_late = tn + fp
total_ontime = fn + tp

metrics = {
    'Overall accuracy':                 f'{accuracy_score(y_test, y_pred)*100:.1f}%',
    'Late submissions caught (recall)': f'{tn/(tn+fp)*100:.1f}%  ({tn}/{total_late})',
    'False flag rate':                  f'{fn/(fn+tp)*100:.1f}%  ({fn}/{total_ontime})',
    'Accuracy — tier >90%':             f'{results[results["prior_tier"]==">90%"]["correct"].mean()*100:.1f}%',
    'Accuracy — tier 60–75%':           f'{results[results["prior_tier"]=="60–75%"]["correct"].mean()*100:.1f}%',
    'Accuracy gap (max - min tier)':    '',
    'Students appealing predictions':   '0  (appeal mechanism not yet implemented)',
    'Manual overrides by teachers':     'N/A  (not yet tracked)',
}

accs_by_tier = [results[results['prior_tier']==t]['correct'].mean()
                for t in ['<60%','60–75%','75–90%','>90%']
                if len(results[results['prior_tier']==t]) > 0]
metrics['Accuracy gap (max - min tier)'] = f'{(max(accs_by_tier)-min(accs_by_tier))*100:.1f} percentage points'

for key, val in metrics.items():
    flag = '  ⚠️' if any(x in key for x in ['gap', 'False flag', 'caught']) else ''
    print(f'  {key:<44} {val}{flag}')

print()
print('Suggested policy: This report should be generated and reviewed each semester.')
print('Trigger for suspension: accuracy gap > 25pp OR late recall < 10% for any tier.')

---

## Part 4: Drafting the Governance Policy

You now have all the data you need. The following is a policy template. Your task is to complete each section — replacing the `[bracketed placeholders]` with your own decisions, grounded in the evidence from Modules 1–4 and the principles of Catholic social teaching.

There are no single correct answers. The goal is a reasoned, internally consistent policy that honestly addresses what the model can and cannot do, protects the most vulnerable stakeholders, and preserves human oversight and accountability.

---

# Data Governance Policy
## Homework Completion Prediction System

**Version:** 1.0 — Draft  
**Author:** [Your name]  
**Date:** [Today's date]  
**Status:** For class review

---

### 1. Purpose and Scope

**1.1 Purpose.** This policy governs the operation of a machine learning system (the "System") that generates predictions about the likelihood that a student will submit a homework assignment on time. The System is intended to [*describe the intended use — e.g., help teachers identify students who may benefit from early check-ins*].

**1.2 What the System may influence.** The System's predictions may inform [*describe allowed uses — e.g., teacher outreach decisions*]. The System's predictions may NOT be used to [*describe prohibited uses — e.g., grade adjustments, disciplinary action, permanent record notation without human review*].

**1.3 Scope.** This policy applies to all staff, administrators, and third-party contractors who access the System's outputs.

**1.4 Catholic social teaching basis.** This policy is grounded in the principle that [*cite one principle — e.g., the human person is always the principle, subject, and purpose of social institutions (Compendium, no. 107)*], and in the obligation to ensure that technology serves the most vulnerable rather than reinforcing existing disadvantage.

---

### 2. Access and Permissions

| Role | What they may see | Conditions |
|---|---|---|
| Class teacher | [*your decision*] | [*e.g., only for their own students; must complete training*] |
| School counselor | [*your decision*] | [*e.g., only when referred by teacher*] |
| School administration | [*your decision*] | [*e.g., aggregate statistics only — no individual student scores*] |
| Student (own record) | [*your decision*] | [*e.g., yes / no — and if yes, what format*] |
| Parent / guardian | [*your decision*] | [*e.g., upon request; with explanation*] |
| Third parties (researchers, vendors) | [*your decision*] | [*e.g., anonymized aggregate data only*] |

**Rationale for access decisions:** [*Write 2–3 sentences explaining the principle behind your access structure. Which Catholic principle guided who should and should not have access?*]

---

### 3. Prediction Threshold and Confidence Requirements

**3.1 Action threshold.** The System will flag a student for teacher review when the predicted probability of on-time submission falls below [*your threshold, e.g., 0.70*]. At this threshold, approximately [*number*]% of students are flagged, of whom approximately [*precision*]% are actually late.

**3.2 Confidence bands.** The System will classify predictions into the following bands for display purposes:

| Probability band | Display label | Recommended teacher response |
|---|---|---|
| prob < 0.50 | [*your label*] | [*e.g., Direct check-in required within 24 hours*] |
| 0.50–0.70 | [*your label*] | [*your recommendation*] |
| 0.70–0.85 | [*your label*] | [*your recommendation*] |
| prob ≥ 0.85 | [*your label*] | [*your recommendation*] |

**3.3 Automated actions.** [*State clearly: are any automated actions permitted without human review? If not, state this explicitly. If yes, describe precisely what is automated and what requires human judgment.*]

**Empirical basis:** [*Reference the precision data from Part 3 — e.g., "At the 0.70 threshold, precision for the late class is 50%, meaning approximately half of all flagged students are on time. This means teacher judgment is essential before any response is taken."*]

---

### 4. Human Oversight Requirements

**4.1 Required human review.** Before any action is taken in response to a System flag — including teacher outreach, counselor referral, or parental notification — [*describe what human review is required, e.g., the classroom teacher must review the student's recent submission history and make an independent judgment*].

**4.2 Documentation.** When a teacher reviews and acts on a System flag, they must document [*e.g., what action was taken, whether the AI prediction influenced the decision, and what the outcome was*]. This documentation serves as the accountability record.

**4.3 Override authority.** Any teacher may override the System's flag for any student at any time, without requiring justification. The System advises; the teacher decides.

**Catholic social teaching basis:** *Magnifica Humanitas* (no. 199) holds that "the decision to use [consequential action] cannot be delegated to opaque or automated processes, but must remain under effective, self-aware and responsible human control."

---

### 5. Disclosure and Transparency

**5.1 Student disclosure.** [*Describe what students will be told about the system — e.g., students will be informed in writing at the start of the term that predictive analytics tools are used, what data they use, and how predictions may influence teacher outreach. OR: students will not be shown their individual scores. Justify your choice.*]

**5.2 Family disclosure.** [*Describe what parents/guardians will be told, and when.*]

**5.3 Teacher training.** All teachers using the System must complete training that covers: (a) how the model works, (b) its documented accuracy and bias findings, (c) the 33-percentage-point accuracy gap between the highest- and lowest-performing student tiers, and (d) the obligation to exercise independent judgment before acting on any flag.

**5.4 Annual public report.** The school will publish an annual summary of System performance, including overall accuracy, bias audit results by student tier, and the number of teacher overrides recorded. This report will be available to families upon request.

---

### 6. Appeal and Redress Mechanisms

**6.1 Student appeal.** Any student who believes a System prediction has unfairly influenced how they are being treated may [*describe the appeal process — e.g., request a meeting with their teacher and a counselor, at which the prediction and its basis must be explained in plain language, and which must result in a documented response within five school days*].

**6.2 Correction mechanism.** If a teacher discovers that a flagged student was on time and the flag was incorrect, [*describe what happens — e.g., the flag is removed from all records; the incident is logged for the next audit cycle*].

**6.3 No adverse record.** Being flagged by the System — even incorrectly — shall not be noted in any permanent student record. Flags are operational tools for teacher decision-making, not assessments of student character or capability.

---

### 7. Monitoring and Audit Schedule

**7.1 Metrics to monitor.** The System will be evaluated each semester on the following metrics:
- Overall accuracy
- Late submission recall (fraction of truly late submissions caught)
- False flag rate (fraction of on-time submissions wrongly flagged)
- Accuracy by prior completion rate tier (the bias audit measure from Module 4)
- Accuracy gap between highest- and lowest-performing tiers
- Number of teacher overrides
- Number of student appeals

**7.2 Audit responsibility.** [*Name who conducts the audit — e.g., a designated data steward or external reviewer*].

**7.3 Retraining schedule.** The model will be retrained with updated data at [*e.g., the start of each academic year*], and the bias audit will be re-run on the retrained model before it is deployed.

---

### 8. Sunset and Revocation Conditions

**8.1 Automatic suspension triggers.** The System will be suspended immediately, pending review, if any of the following conditions are detected in a monitoring report:
- Overall late recall falls below [*your threshold, e.g., 10%*] — the model is no longer providing useful signal
- Accuracy gap between tiers exceeds [*your threshold, e.g., 30 percentage points*] — bias has become unacceptably large
- [*Add your own condition — e.g., more than X% of teacher overrides in a term suggests the model is not aligned with actual risk*]

**8.2 Discretionary revocation.** The system may be suspended at the discretion of school leadership if [*describe circumstances — e.g., a student or family demonstrates concrete harm resulting from a System flag*].

**8.3 End-of-life.** This System will be formally reviewed for continued deployment at the end of each academic year. Continued deployment requires a positive finding on all monitoring metrics and confirmation that no better alternatives are available. Silence is not consent — the system must be actively re-approved.

---

### Acknowledgment of Known Limitations

By deploying this System under this policy, the school acknowledges the following:

1. The System catches approximately [*X*]% of late submissions at the selected threshold, meaning the majority of late submissions will not be flagged.
2. The System performs significantly worse for students with prior completion rates below 75%, who are also the students most likely to be late.
3. The System cannot account for circumstances not present in the data: illness, family crises, learning differences, or personal change.
4. No prediction should be treated as a diagnosis, a judgment of student character, or a substitute for teacher-student relationship.
5. The System's use of `prior_completion_rate` as a feature means students who have struggled in the past will be flagged more often, potentially reinforcing the very disadvantage the school aims to address.

---

### Summary: Catholic Social Teaching Alignment

| Principle | How This Policy Applies It |
|---|---|
| Human dignity (*imago Dei*) | [*write one sentence*] |
| Common good | [*write one sentence*] |
| Subsidiarity | [*write one sentence*] |
| Solidarity / preferential option | [*write one sentence*] |
| Accountability | [*write one sentence*] |

---

## Part 5: The Hardest Question — Should This System Be Deployed at All?

You have now built the system, evaluated it, audited it for bias, and drafted a policy to govern it. Before completing the policy, there is one more question to confront honestly: **given everything you have found, should this system be deployed?**

This is not a trick question. There are legitimate arguments on both sides.

In [ ]:
# Generate a final decision support summary
print('=' * 65)
print('DEPLOYMENT DECISION SUPPORT SUMMARY')
print('=' * 65)
print()
print('THE CASE FOR DEPLOYMENT (with strong governance):')
print()
print('  + At threshold 0.70, catches 43% of late submissions')
print('    (vs. 0% with no system at all)')
print('  + Very high risk band (prob < 0.50) has 50% precision —')
print('    better than random, and these students genuinely need attention')
print('  + With proper human oversight, even an imperfect signal can')
print('    prompt a teacher to check in who otherwise would not')
print('  + Total cost: a teacher reviews ~12 flagged students per 100')
print('    assignments, of whom ~6 are genuinely at risk')
print()
print('THE CASE AGAINST DEPLOYMENT:')
print()
print('  - For the students who most need help (prior rate <75%),')
print('    the model is 37–50% accurate — worse than useless as a guide')
print('  - At threshold 0.70, half the flagged students are on time;')
print('    teacher attention is misdirected more than it is focused')
print('  - Using prior_completion_rate as a feature embeds the past')
print('    into the future, making it harder for struggling students')
print('    to escape a label once assigned')
print('  - The model cannot tell you why — only whether. A teacher')
print('    already knows who is struggling; what they need is not a')
print('    prediction but time and resources to respond')
print()
print('=' * 65)
print('YOUR RECOMMENDATION: [Write in the cell below]')
print('=' * 65)

### Your Deployment Recommendation

Write a 3–5 sentence recommendation. State clearly:
1. Whether you recommend deployment, limited deployment, or no deployment
2. If deployment: under which scenario (A, B, C, or D), at which threshold, and with which governance conditions
3. Your primary reason, grounded in at least one Catholic social teaching principle
4. What would change your recommendation — what additional evidence, what changed circumstances

*[Write your recommendation here]*

---

## Summary and What's Coming Next

### What You Accomplished in This Module

- Completed a full stakeholder analysis identifying every party affected by the system
- Analyzed four deployment scenarios and evaluated each on governance dimensions
- Used empirical risk band data to make an informed, specific threshold recommendation
- Drafted a complete eight-element data governance policy grounded in both technical evidence and Catholic social teaching
- Confronted the hardest question — whether to deploy at all — with evidence and principle

### Coming in Module 6 (Classes 11–12)

Module 6 is the final technical module. You will build a **presentation-ready report** summarizing the entire project — what was built, what was found, what the bias audit revealed, and what the governance policy recommends. The report will include:

- An executive summary suitable for a school board or parent audience
- Visualizations from Modules 1–5 (polished for presentation)
- The complete governance policy from this module
- A final reflection connecting the technical findings to Catholic teaching on human dignity, the common good, and the care owed to the most vulnerable

This report becomes the core of your Class 13 final project presentation.

---

> **A final thought for this module:** *Magnifica Humanitas* (no. 111) states that "every design choice reflects a vision of humanity." The governance policy you just drafted is also a design choice — it is, in fact, among the most important design choices in any AI project. The algorithm produces predictions. The governance policy determines whether those predictions help people or harm them. That makes the policy at least as important as the model — and makes the person who writes it at least as responsible for the system's effects as the person who trained it.